In [1]:
# core
import os
import numpy as np
import pandas as pd

# modeling
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split, GridSearchCV, KFold, cross_val_predict, cross_val_score

# metrics
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# plots
import matplotlib.pyplot as plt
import seaborn as sns

# utils
from math import sqrt

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


In [2]:
# path to your training data
data_path = "./data/train_real.csv"  # change if needed
df = pd.read_csv(data_path)

# columns to exclude from features (same as RF script)
non_feature_cols = {
    "mxene","gibbs_free_energy","composition_obj"
}

# feature columns = all except the excluded ones (then keep only numeric)
feature_cols = [c for c in df.columns if c not in non_feature_cols]
X_all = df[feature_cols].select_dtypes(include=[np.number]).copy()

# label (same target as RF version)
target_col = "gibbs_free_energy"
y_all = df[target_col].astype(float).copy()

# basic checks
print("n_samples:", len(df))
print("n_features (numeric):", X_all.shape[1])
print("first 5 feature columns:", feature_cols[:5])
X_all.head()


n_samples: 538
n_features (numeric): 250
first 5 feature columns: ['H', 'He', 'Li', 'Be', 'B']


,H,He,Li,Be,B,C,N,O,F,Ne,...,MagpieData range GSmagmom,MagpieData mean GSmagmom,MagpieData avg_dev GSmagmom,MagpieData mode GSmagmom,MagpieData minimum SpaceGroupNumber,MagpieData maximum SpaceGroupNumber,MagpieData range SpaceGroupNumber,MagpieData mean SpaceGroupNumber,MagpieData avg_dev SpaceGroupNumber,MagpieData mode SpaceGroupNumber
0,0.0,0.0,0.0,0.0,0.2,0.0,0.0,0.0,0.4,0.0,...,0.0,0.0,0.0,0.0,15.0,229.0,214.0,130.8,92.64,15.0
1,0.0,0.0,0.0,0.0,0.0,0.2,0.0,0.0,0.4,0.0,...,0.0,0.0,0.0,0.0,15.0,229.0,214.0,129.4,91.52,15.0
2,0.0,0.0,0.0,0.0,0.0,0.2,0.0,0.0,0.4,0.0,...,0.0,0.0,0.0,0.0,15.0,229.0,214.0,136.4,97.12,15.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.2,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,64.0,229.0,165.0,156.0,73.60,64.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.2,0.4,0.0,0.0,...,0.0,0.0,0.0,0.0,12.0,229.0,217.0,135.2,98.56,12.0


In [3]:
X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.2, random_state=RANDOM_STATE
)

# simple imputation (median) to handle any missing values
X_train = X_train.fillna(X_train.median(numeric_only=True))
X_test  = X_test.fillna(X_train.median(numeric_only=True))  # use train medians

print("train shape:", X_train.shape, "| test shape:", X_test.shape)


train shape: (430, 250) | test shape: (108, 250)


In [4]:
# choose fast/robust tree method
# "hist" is fast on CPU; switch to "gpu_hist" if you know a CUDA GPU is available
TREE_METHOD = "hist"

xgb = XGBRegressor(
    random_state=RANDOM_STATE,
    n_estimators=500,          # will be overridden by grid when applicable
    learning_rate=0.1,
    max_depth=8,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_lambda=1.0,
    reg_alpha=0.0,
    min_child_weight=1.0,
    objective="reg:squarederror",
    n_jobs=1,
    tree_method=TREE_METHOD,
)

# a compact but effective grid; adjust as needed
param_grid = {
    "n_estimators": [400, 800],
    "max_depth": [6, 10],
    "learning_rate": [0.05, 0.1],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.7, 1.0],
    "min_child_weight": [1, 3],
    "reg_lambda": [1.0, 5.0],
    "reg_alpha": [0.0, 0.5],
}

cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

gscv = GridSearchCV(
    estimator=xgb,
    param_grid=param_grid,
    scoring="neg_root_mean_squared_error",
    cv=cv,
    n_jobs=-1,
    verbose=1,
    return_train_score=True
)

gscv.fit(X_train, y_train)
best_xgb = gscv.best_estimator_
print("best params:", gscv.best_params_)
print("best cv RMSE:", -gscv.best_score_)


Fitting 5 folds for each of 256 candidates, totalling 1280 fits
best params: {'colsample_bytree': 1.0, 'learning_rate': 0.1, 'max_depth': 10, 'min_child_weight': 1, 'n_estimators': 400, 'reg_alpha': 0.0, 'reg_lambda': 5.0, 'subsample': 0.8}
best cv RMSE: 0.599770462776516


In [5]:
# Predict on new feature-only CSV

# Path setup
new_features_path = "./data/predict_800.csv"        # input file
out_path = "./data/pred_gibbs_free.csv"             # output file

new_df = pd.read_csv(new_features_path)

# Keep a copy of the mxene column before dropping
if "mxene" not in new_df.columns:
    raise ValueError("Input file must contain a 'mxene' column to identify entries.")
mxene_col = new_df["mxene"].copy()

# Drop non-feature columns (same as training exclusion list)
NON_FEATURE_COLS = {"mxene", "composition_obj"}
new_df = new_df.drop(columns=[c for c in new_df.columns if c in NON_FEATURE_COLS],
                     errors="ignore")

# Ensure same feature order as training
missing = set(X_train.columns) - set(new_df.columns)
extra   = set(new_df.columns) - set(X_train.columns)

if missing:
    raise ValueError(f"The following required feature columns are missing in the input: {sorted(list(missing))[:10]} ...")

# Align columns and fill missing with NaN
X_new = new_df.reindex(columns=X_train.columns, fill_value=np.nan)

# Impute with training medians
X_new = X_new.fillna(X_train.median(numeric_only=True))

# --- Prediction (use your trained model) ---
pred = best_xgb.predict(X_new)

# --- Write output: only mxene + prediction ---
pred_df = pd.DataFrame({
    "mxene": mxene_col,
    "predicted_gibbs_free_energy": pred
})

pred_df.to_csv(out_path, index=False)
print(f"Saved predictions to: {out_path}")
pred_df.head()


Saved predictions to: ./data/pred_gibbs_free.csv


,mxene,predicted_gibbs_free_energy
0,Cr2CBr2,2.102955
1,Hf2CBr2,2.286548
2,Mo2CBr2,2.109220
3,Nb2CBr2,2.174036
4,Sc2CBr2,2.141488
